# boundary-detector: train the tool-output injection detector (GPU)

Fine-tunes `microsoft/deberta-v3-xsmall` into a `BENIGN`/`INJECTION` classifier for the boundary
guard. Runtime → **Change runtime type → T4 GPU** (free), then run the cells top to bottom.

1. Upload `train.jsonl` and `val.jsonl` from `packages/detector/data/` (generate them with
   `uv run boundary-detector data` in the repo).
2. Train (a few minutes on a T4).
3. Download `boundary-detector-model.zip`, unzip it into `packages/detector/model/` in the repo, then run
   `uv run boundary-detector evaluate packages/detector/model`.


In [ ]:
!pip -q install "transformers>=4.46,<5" "datasets>=3.0" "accelerate>=1.0" \
  "scikit-learn>=1.5" "sentencepiece>=0.2" "protobuf>=5.0"
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# Upload train.jsonl and val.jsonl from packages/detector/data/
from google.colab import files
up = files.upload()
assert "train.jsonl" in up and "val.jsonl" in up, "please upload both train.jsonl and val.jsonl"


In [ ]:
import json
from pathlib import Path

BASE_MODEL = "microsoft/deberta-v3-xsmall"
EPOCHS = 3
BATCH = 16
MAX_LEN = 512
SEED = 20260926
ID2LABEL = {0: "BENIGN", 1: "INJECTION"}; LABEL2ID = {v: k for k, v in ID2LABEL.items()}

def load(p):
    return [json.loads(l) for l in Path(p).read_text().splitlines() if l.strip()]
train_rows, val_rows = load("train.jsonl"), load("val.jsonl")
print("train", len(train_rows), "val", len(val_rows),
      "| positives", sum(r["label"] for r in train_rows))

In [ ]:
import numpy as np
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
set_seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_ds(rows):
    ds = Dataset.from_dict({"text":[r["text"] for r in rows], "labels":[r["label"] for r in rows]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])
train_ds, val_ds = to_ds(train_rows), to_ds(val_rows)

def metrics(ep):
    logits, labels = ep; pred = np.argmax(logits, -1)
    return {"f1": f1_score(labels, pred, zero_division=0),
            "precision": precision_score(labels, pred, zero_division=0),
            "recall": recall_score(labels, pred, zero_division=0)}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)
args = TrainingArguments(output_dir="ckpt", num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH*2,
    learning_rate=2e-5, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", greater_is_better=True, logging_steps=25, seed=SEED, report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

In [ ]:
# Save and download the model
out = "detector-model"; trainer.save_model(out); tok.save_pretrained(out)
import shutil; shutil.make_archive("boundary-detector-model", "zip", out)
from google.colab import files; files.download("boundary-detector-model.zip")